# 01 — Data Exploration

**AutoClaim AI** | IE University Deep Learning Final Project

This notebook inventories the CarDD dataset, checks image quality, and prepares the folder structure used by all subsequent notebooks.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))
import config
print("Project root:", config.BASE_DIR)

## 1. Dataset Overview

The **CarDD** dataset uses COCO format (JSON annotations + raw images).
- Already split: train (2 816) / val (810) / test (374)
- 6 damage categories: dent, scratch, crack, glass shatter, lamp broken, tire flat

Since images can have multiple damage annotations, we assign each image the **dominant class** — the category with the largest total annotation area. This converts COCO instance-segmentation labels into a single-label classification problem.

In [ ]:
import json, os
from collections import defaultdict, Counter

def load_coco_summary(ann_path):
    with open(ann_path) as f:
        data = json.load(f)
    cats = {c["id"]: c["name"] for c in data["categories"]}
    img_area = defaultdict(lambda: defaultdict(float))
    for ann in data["annotations"]:
        img_area[ann["image_id"]][ann["category_id"]] += ann["area"]
    img_primary = {iid: max(a, key=a.get) for iid, a in img_area.items()}
    counts = Counter(cats[cid] for cid in img_primary.values())
    return dict(counts)

for split in ["train", "val", "test"]:
    ann = os.path.join(config.COCO_ANN_DIR, f"instances_{split}2017.json")
    counts = load_coco_summary(ann)
    print(f"\n{split.upper()} split:")
    for cls, cnt in sorted(counts.items()):
        print(f"  {cls:>14}: {cnt:>4}")

### Actual class distribution (after COCO → dominant-class conversion)

| Class | Train | Val | Test | Total |
|-------|------:|----:|-----:|------:|
| crack | 61 | 19 | 8 | 88 |
| dent | 726 | 207 | 109 | 1 042 |
| glass shatter | 358 | 101 | 65 | 524 |
| lamp broken | 247 | 74 | 42 | 363 |
| scratch | 932 | 268 | 122 | 1 322 |
| tire flat | 492 | 141 | 28 | 661 |
| **TOTAL** | **2 816** | **810** | **374** | **4 000** |

**Key imbalance**: `scratch` (932) has **15×** more training images than `crack` (61).  
This is handled in `src/train.py` via capped class weights (max 2.5×) — see `config.CLASS_WEIGHT_MAX`.

## 2. Prepare Processed Dataset

In [ ]:
from src.data_utils import prepare_dataset
summary = prepare_dataset()  # skips if already done
print("Summary:", summary)

## 3. Class Distribution Plot

In [ ]:
from src.data_utils import plot_class_distribution
import os
os.makedirs(config.FIG_DIR, exist_ok=True)
plot_class_distribution(summary, save_path=os.path.join(config.FIG_DIR, "class_distribution.png"))

## 4. Sample Images per Class

In [ ]:
from src.data_utils import plot_sample_images
plot_sample_images(n_per_class=4, save_path=os.path.join(config.FIG_DIR, "sample_images.png"))

## 5. Image Quality Check

In [ ]:
from src.data_utils import check_corrupted_images
bad = check_corrupted_images()
print(f"Corrupted images: {len(bad)}")

## 6. Data Inventory Report

In [ ]:
import pandas as pd
rows = []
for split, counts in summary.items():
    for cls, cnt in counts.items():
        rows.append({"split": split, "class": cls, "count": cnt})
df = pd.DataFrame(rows)
pivot = df.pivot(index="class", columns="split", values="count").fillna(0).astype(int)
pivot["total"] = pivot.sum(axis=1)
pivot.loc["TOTAL"] = pivot.sum()
print(pivot)
os.makedirs(config.REPORTS_DIR, exist_ok=True)
with open(os.path.join(config.REPORTS_DIR, "data_inventory.md"), "w") as f:
    f.write("# Data Inventory\n\n")
    f.write(pivot.to_markdown())
    f.write("\n\n*Generated by 01_data_exploration.ipynb*\n")
print("Saved data_inventory.md")